# 02 — Mortality Prediction

Predict in-hospital mortality using a Transformer model.
Same 4-step PyHealth pipeline as readmission prediction.

**Task**: Binary (died in hospital: yes/no)  
**Model**: Transformer  
**Metrics**: PR-AUC, ROC-AUC, F1

In [ ]:
from pyhealth_enterprise.datasets.synthetic import SyntheticEHRDataset

ds = SyntheticEHRDataset()
ds.load()

In [ ]:
from pyhealth.tasks import mortality_prediction_mimic3_fn
from pyhealth.datasets import split_by_patient, get_dataloader

task_dataset = ds.dataset.set_task(mortality_prediction_mimic3_fn)
train, val, test = split_by_patient(task_dataset, [0.8, 0.1, 0.1])
train_loader = get_dataloader(train, batch_size=32, shuffle=True)
val_loader   = get_dataloader(val,   batch_size=32, shuffle=False)
test_loader  = get_dataloader(test,  batch_size=32, shuffle=False)

In [ ]:
from pyhealth.models import Transformer
from pyhealth.trainer import Trainer

model = Transformer(
    dataset=task_dataset,
    feature_keys=['conditions', 'drugs'],
    label_key='mortality',
    mode='binary',
    embedding_dim=128,
    nhead=4,
    num_encoder_layers=2,
    dropout=0.1,
)
trainer = Trainer(model=model, metrics=['pr_auc', 'roc_auc', 'f1'])
trainer.train(train_dataloader=train_loader, val_dataloader=val_loader,
              epochs=50, monitor='pr_auc')
result = trainer.evaluate(test_loader)
print(result)

In [ ]:
# Enterprise pattern
from pyhealth_enterprise.tasks.mortality import setup_mortality_task
from pyhealth_enterprise.models.registry import ModelName, get_model
from pyhealth_enterprise.pipelines.batch_risk_scorer import BatchRiskScorer

train_l, val_l, test_l = setup_mortality_task(ds.dataset)
m = get_model(ModelName.TRANSFORMER, task_dataset, ['conditions', 'drugs'], 'mortality')
scorer = BatchRiskScorer(m, 'transformer_mortality')
scorer.train(train_l, val_l, epochs=10)
risk_df = scorer.score_batch(test_l)
risk_df.head()